# 2-clean&filter

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "ID_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)
df.shape

(1128128, 27)

In [2]:
# # Si test 16eme
# import pandas as pd

# df = pd.read_csv(
#     "../data/interim/extract_16.csv",
#     low_memory=False,
#     dtype={
#         "ID_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
#     },
# )
# df.shape

In [3]:
# # Si test 15ème
# import pandas as pd

# df = pd.read_csv(
#     "../data/interim/extract_15.csv",
#     low_memory=False,
#     dtype={
#         "ID_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
#     },
# )
# df.shape

In [4]:
# Ne garder que le code style NORMAL
df = df[df["Code_style"] == "NORMAL"]

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
# Role_debat n'est pas bien identifié, utiliser Nom_orateur
df = df[~df["Nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# # TODO: AVISER selon introduction 15ème législature
# # Pour le sous cas de la 16 législature : nettoyer le fichier qui n'est pas au bon endroit
# # = date de 2021
# df = df[df["UID"] != "CRSANR5L16S2021O1N144"]
# # -> remonté dès l'extraction en notebook 1.

# Garder une trace de la longueur des interventions brutes
df["len_dirtytext"] = df["Texte"].str.len()

# Stabiliser le ID_orateur pour etre au format AN (pour matcher données)
# marche car pandas propage les NaN quand bien reconnu comme objet
# donc l'importance de str au chargement (et de pas forcer en str après ?)
df["ID_orateur"] = "PA" + df["ID_orateur"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["Code_parole"] = df["Code_parole"].fillna("non_précisé")

df.shape

(683680, 28)

In [5]:
df["ID_orateur"].isna().sum()  # 0

44952

In [6]:
# aperçu des répartitions
df.groupby("Code_parole", dropna=False)["len_dirtytext"].describe()

,count,mean,std,min,25%,50%,75%,max
Code_parole,,,,,,,,
(null),1.0,186.000000,NaN,186.0,186.00,186.0,186.00,186.0
AVIS_COM_1_10,1.0,1278.000000,NaN,1278.0,1278.00,1278.0,1278.00,1278.0
AVIS_COM_1_20,45107.0,450.390582,482.536522,3.0,118.50,315.0,618.00,8179.0
AVIS_GVT_1_20,38454.0,486.647371,681.527170,4.0,17.00,240.0,685.00,11340.0
PAROLE_1_1,21.0,556.761905,1080.603346,30.0,101.00,241.0,529.00,5088.0
PAROLE_1_2,247778.0,974.519219,1294.330875,3.0,245.00,547.0,1207.00,68092.0
PRESIDE_DISCOURS_1_10,2.0,95.500000,92.630988,30.0,62.75,95.5,128.25,161.0
Raccroche_apres_inter,9.0,1638.666667,1450.025431,60.0,154.00,1340.0,2316.00,3939.0
non_précisé,352287.0,282.342874,625.719777,1.0,17.00,37.0,251.00,25609.0


In [7]:
# TODO: regrouper les interventions interrompues ?

**?????????aviser pour regrouper les interventions interrompues ?????????**

## Match députés

### Match infos générales (historique)

In [8]:
df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")
# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(columns=["mail", "twitter", "facebook", "website"])


In [9]:
print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merge et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="ID_orateur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion:", df.shape)

shape avant fusion: (683680, 28)
shape après fusion: (683680, 50)


### Match temporel des affiliations

In [10]:
# recodage des grandes dénominations des groupes
# (moins sensible aux évolutions marginales de dénomination)

df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format degeu

# Recoder les partis pour stabilité temporelle des noms
# TODO: visiblement d'autres : soc-a, agir-e,fi/lfi, UDI/modem, etc.
# Vérifier
recodage = {
    "RE": "REN",
    "LAREM": "REN",
    "DEM": "MODEM",
    "SOC": "PS",
    "NG": "PS",
    "LFI-NUPES": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "PCF",
    "GDR": "PCF",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
}

df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(recodage)


In [11]:
# Si besoin de réexplorer les répartitions :

# df_affiliation["libelleAbrev"].value_counts()
# counts_abrev = df_affiliation["libelleAbrev"].value_counts()
# counts_abrege = df_affiliation["libelleAbrege"].value_counts()
# counts_libelle = df_affiliation["libelle"].value_counts()
# counts_recod = df_affiliation["parti_recod"].value_counts()

# counts_df = (
#     pd.DataFrame(
#         {
#             "libelleAbrev_count": counts_abrev,
#             "libelleAbrege_count": counts_abrege,
#             "libelle_count": counts_libelle,
#             "libelle_recod_count": counts_recod,
#         }
#     )
#     .fillna(0)
#     .astype(int)
# )

# # counts_df.to_csv("group_counts.csv")
# counts_df

In [12]:
##############################################################
# JE GARDE LE TEMPS QUE MATTHIAS PUISSE VOIR MA MAUVAISE IDÉE
# PASSÉ À UN LOOKUP ET RENVOI AFFILIATION TEMPORELLE VALIDE
##############################################################

# # EN cours : créer une fonction de renvoi du parti dans le temps
# # Et donc galérer avec les dates d'intervention vs date de début et fin affiliation ?

# # TODO: trouver ce qui merde car pour l'instant fait avec les pieds
# # pistes : cas des sans id_orateur ? cas des interv sans date ?
# # souci : tous ceux qui ont pas d'ID député > pas de renvoi de date début ou fin, etc.
# # Changer la logique ? > au final pas gros fichier
# # On peut partir sur lookup


# print("shape avant merge: ", df.shape)
# # Garder uniquement les colonnes utiles
# df_affiliation = df_affiliation[["mpId", "dateDebut", "dateFin", "parti_recod"]].copy()

# # Conversion des dates affiliation en datetime
# df_affiliation["dateDebut"] = pd.to_datetime(
#     df_affiliation["dateDebut"], errors="raise"
# )
# df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")


# # Conversion date intervention
# df["DateSeance_ts"] = pd.to_datetime(df["DateSeance"], format="%Y%m%d%H%M%S%f")

# # merge sur l'identifiant (ID_orateur vs mpId)
# df_merged = df.merge(df_affiliation, left_on="ID_orateur", right_on="mpId", how="left")

# # Garder uniquement les affiliations valides à la date de l’intervention
# df_match_affiliation = df_merged[
#     (df_merged["DateSeance_ts"] >= df_merged["dateDebut"])
#     & (df_merged["DateSeance_ts"] <= df_merged["dateFin"])
# ]

# print("shape après merge:", df_match_affiliation.shape)

In [13]:
# missing_ids = set(df["ID_orateur"]) - set(df_affiliation["mpId"])
# print(len(missing_ids), "orateurs n'ont aucune affiliation connue")

In [14]:
# mask = ~(
#     (df_merged["DateSeance_ts"] >= df_merged["dateDebut"]) &
#     (df_merged["DateSeance_ts"] <= df_merged["dateFin"])
# )
# print("interventions hors période:", mask.sum())

In [15]:
# TODO: OUPS WE GOT A PROBLEM WITH THE FUSION (works on 16th, works on 15th, but not on both))
# BAH finalement ça marche ? j'ai été assez neuneu pour relancer une cellule ?
# Et genre foirer les ID en ajoutant 2 fois "PA" ?
# we love notebooks…………………

In [16]:
# Plutôt qu'un merge foireux parti sur un lookup ligne‑à‑ligne
# (= pb des orateurs non députés qui étaient pas présents, etc.)
# Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb


# préparation des dates
df["DateSeance_ts"] = pd.to_datetime(
    df["DateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# # aviser si jamais besoin traiter affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


def get_parti_for_row(row):
    mp = row.get("ID_orateur")  # correspond au mpId
    # gérer le cas des orateurs non députés
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("DateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        if rec["dateDebut"] <= ts <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# appliquer et marquer les inconnus (pour repérage futur)
df["parti_affiliation"] = df.apply(get_parti_for_row, axis=1)
df["parti_affiliation"] = df["parti_affiliation"].fillna("UNKNOWN")

print(
    "affectés :",
    df["parti_affiliation"].ne("UNKNOWN").sum(),
    "UNKNOWN :",
    (df["parti_affiliation"] == "UNKNOWN").sum(),
)
# TODO: envisager de forcer le renvoi de la derniere affiliation connue de df_deputes ?
# Aviser pour des matchs plus précis (cas limites, etc.) sur la base des repérages de matthias.
# Aviser le cas des UNKNOWN = EPR et fusion REN ?

affectés : 529995 UNKNOWN : 153685


In [17]:
# # SI besoin de recomprendre la logique
# # de la récupération des affiliations :

# # type et aperçu
# for mp, g in df_affiliation.groupby("mpId"):
#     print(mp, type(g))
#     print(g.head())
#     break

# # ou inspecter le résultat stocké
# print(type(aff_by_mp["PA795864"]))  # list
# print(aff_by_mp["PA795864"][:2])  # 2 premiers enregistrements (dicts)

In [18]:
df["parti_affiliation"].value_counts().sort_index()

parti_affiliation
AGIR-E       2334
ECO         14027
EDS           917
FI          38240
HOR          5053
LFI         36484
LIOT        16738
LR         123720
MODEM       36759
NI          12966
PCF         40135
PS          36972
REN        122538
RN          21684
SOC-A        4179
UDI         17249
UNKNOWN    153685
Name: count, dtype: int64

In [19]:
df["groupeAbrev"].value_counts().sort_index()

groupeAbrev
AGIR-E        4274
DEM          47189
DR           65347
ECOLO          144
ECOS         23834
EPR          70220
FI            6175
GDR          23002
GDR-NUPES    16755
HOR          13846
LAREM        47459
LES-REP       3307
LFI-NFP      50463
LFI-NUPES     9374
LIOT         19604
LR           45043
LT            5533
MODEM           43
NG             138
NI           25228
RE           44629
RN           23592
SOC          31755
SOC-A         5659
UDI-AGIR       182
UDI_I         5788
UDR           1562
UMP            207
Name: count, dtype: int64

## Export

In [20]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# import csv  # pour utiliser csv.QUOTE_ALL et résoudre le soucis d'écart.
# df.to_csv(
#     "../data/interim/data_cleaning.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL,  # permet de résoudre le soucis
# )

In [28]:
# verif ecriture/lecture ok
print("df shape:", df.shape)

df_test = pd.read_csv("../data/interim/data_cleaning.csv", low_memory=False)

print("df_test shape (après export import): ", df_test.shape)

df shape: (683680, 52)
df_test shape (après export import):  (683680, 52)


In [23]:
# TODO: regrouper les interventions interrompues ?